# Gradient Accumulation High-Batch Experiment (Full)

Bu notebook sadece **gradient accumulation** ile kucuk+büyük effective batch size degerlerini dener ve karsilastirma raporu uretir.

## 1. Ortam Kurulumu ve Importlar

Bu hucre proje kokunu bulur, `gradient_accumulation` yolunu `sys.path` icine alir, gerekli kutuphaneleri import eder ve **dogru train modulu** yuklenmis mi kontrol eder.

In [ ]:
from pathlib import Path
import os
import sys
import json
import random
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

sns.set_theme(style="whitegrid")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for parent in [current, *current.parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError("Project root not found (no .git folder).")

PROJECT_ROOT = find_project_root()
GA_PATH = (PROJECT_ROOT / "gradient_accumulation").resolve()
if str(GA_PATH) not in sys.path:
    sys.path.insert(0, str(GA_PATH))

from functions.dataset import (
    COVIDCXNetDataset,
    DataLoaderConfig,
    build_transforms,
    create_dataloader,
)
from functions.train import TrainConfig, fit
import functions.train as train_module

expected_train_path = (GA_PATH / "functions" / "train.py").resolve()
actual_train_path = Path(train_module.__file__).resolve()
assert actual_train_path == expected_train_path, (
    f"Wrong train module imported. Expected: {expected_train_path}, got: {actual_train_path}"
)
print(f"[OK] Imported train module from: {actual_train_path}")

## 2. Konfigurasyon ve Profil Secimi

Bu hucre JSON config dosyasini okur, `full` profilini aktif eder, cihaz secimini yapar ve calisacak EBS listesini ekrana yazar.

In [ ]:
PROFILE_NAME = "full"
CONFIG_PATH = PROJECT_ROOT / "batchsize" / "configs" / "gradient_accumulation_experiment.json"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    CONFIG = json.load(f)

PROFILE = CONFIG["profiles"][PROFILE_NAME]
RUNS_ROOT = Path(CONFIG["runs_root"])
if not RUNS_ROOT.is_absolute():
    RUNS_ROOT = (PROJECT_ROOT / RUNS_ROOT).resolve()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Profile: {PROFILE_NAME}")
print(f"Device: {device}")
print(f"Runs root: {RUNS_ROOT}")
print(f"Model: {CONFIG['model_name']}")
print(f"Effective batch sizes: {PROFILE['effective_batch_sizes']}")

## 3. Reproducibility (Seed Ayarlari)

Bu hucre onceki batch size calismasiyla ayni seed duzenini uygular (`random`, `numpy`, `torch`, `cudnn`). Boylece kosular arasi tekrar edilebilirlik korunur.

In [ ]:
def set_seed_like_previous_experiment(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = bool(deterministic)
    torch.backends.cudnn.benchmark = not bool(deterministic)

set_seed_like_previous_experiment(CONFIG["seed"], CONFIG["deterministic"])
print("[OK] Seeds configured like batch_size_experiment_notebook.ipynb")

## 4. Veri Seti Hazirlama

Bu hucre train/validation transformlarini kurar, `COVIDCXNetDataset` nesnelerini olusturur ve val split adaylarini sirasiyla deneyip uygun olanini secer.

In [ ]:
train_transform = build_transforms(image_size=CONFIG["image_size"], augment=True)
eval_transform = build_transforms(image_size=CONFIG["image_size"], augment=False)

train_dataset = COVIDCXNetDataset(
    csv_file=CONFIG["data_csv_file"],
    root_dir=CONFIG["data_root_dir"],
    transform=train_transform,
    split="train",
)

val_dataset = None
selected_val_split = None
for split_name in CONFIG["val_split_candidates"]:
    try:
        val_dataset = COVIDCXNetDataset(
            csv_file=CONFIG["data_csv_file"],
            root_dir=CONFIG["data_root_dir"],
            transform=eval_transform,
            split=split_name,
        )
        selected_val_split = split_name
        break
    except ValueError:
        continue

if val_dataset is None:
    raise ValueError("No validation split found in configured candidates.")

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)} (split={selected_val_split})")

## 5. Accumulation Kosularini Calistirma

Bu hucre ResNet50 modelini kurar, her effective batch size icin `accumulation_steps` hesaplar, egitimi calistirir ve kosu bazli metrikleri tek bir `comparison_df` tablosunda toplar.

In [ ]:
def build_resnet50(num_classes: int) -> nn.Module:
    try:
        model = models.resnet50(weights="DEFAULT")
    except Exception:
        model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def run_single_accumulation(effective_batch_size: int) -> dict:
    micro_batch_size = int(PROFILE["micro_batch_size"])

    if effective_batch_size % micro_batch_size != 0:
        return {
            "effective_batch_size": effective_batch_size,
            "micro_batch_size": micro_batch_size,
            "accumulation_steps": None,
            "status": "invalid_config",
            "error": "effective_batch_size is not divisible by micro_batch_size",
        }

    accumulation_steps = effective_batch_size // micro_batch_size
    run_name = f"ga_resnet50_mb{micro_batch_size}_ebs{effective_batch_size}_acc{accumulation_steps}_{PROFILE_NAME}"

    train_loader = create_dataloader(
        train_dataset,
        DataLoaderConfig(
            batch_size=micro_batch_size,
            shuffle=True,
            num_workers=CONFIG["num_workers"],
            drop_last=False,
        ),
        device=device,
    )
    val_loader = create_dataloader(
        val_dataset,
        DataLoaderConfig(
            batch_size=micro_batch_size,
            shuffle=False,
            num_workers=CONFIG["num_workers"],
            drop_last=False,
        ),
        device=device,
    )

    model = build_resnet50(CONFIG["num_classes"])
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=float(CONFIG["optimizer"]["lr"]))

    train_cfg = TrainConfig(
        num_epochs=int(PROFILE["num_epochs"]),
        accumulation_steps=int(accumulation_steps),
        patience=3,
        run_name=run_name,
        output_dir=str(RUNS_ROOT),
        log_every_n_steps=10,
        system_log_interval=50,
        save_best=True,
    )

    try:
        result = fit(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            config=train_cfg,
            logger=None,
            show_progress=True,
        )

        run_dir = Path(result.run_dir)
        epoch_df = pd.read_csv(run_dir / "epoch_metrics.csv")
        system_df = pd.read_csv(run_dir / "system_metrics.csv")
        summary = pd.read_json(run_dir / "run_summary.json", typ="series")

        val_epoch = epoch_df[epoch_df["phase"] == "val"]
        train_epoch = epoch_df[epoch_df["phase"] == "train"]

        return {
            "effective_batch_size": effective_batch_size,
            "micro_batch_size": micro_batch_size,
            "accumulation_steps": accumulation_steps,
            "status": "success",
            "run_name": run_name,
            "run_dir": str(run_dir),
            "best_val_loss": summary.get("best_val_loss", np.nan),
            "final_val_acc": (val_epoch["accuracy"].iloc[-1] if len(val_epoch) else np.nan),
            "total_train_time_sec": summary.get("total_train_time_sec", np.nan),
            "peak_vram_mb": (train_epoch["peak_vram_mb"].max() if len(train_epoch) else np.nan),
            "avg_samples_per_sec": (train_epoch["samples_per_sec"].mean() if len(train_epoch) else np.nan),
            "error": None,
        }
    except torch.cuda.OutOfMemoryError:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            "effective_batch_size": effective_batch_size,
            "micro_batch_size": micro_batch_size,
            "accumulation_steps": accumulation_steps,
            "status": "oom",
            "error": "CUDA OOM",
        }
    except Exception as e:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            "effective_batch_size": effective_batch_size,
            "micro_batch_size": micro_batch_size,
            "accumulation_steps": accumulation_steps,
            "status": "failed",
            "error": f"{type(e).__name__}: {e}",
            "traceback": traceback.format_exc(limit=5),
        }

comparison_rows = []
for ebs in PROFILE["effective_batch_sizes"]:
    print(f"\n[RUN] effective_batch_size={ebs}")
    row = run_single_accumulation(int(ebs))
    comparison_rows.append(row)
    print(f" -> status={row['status']}")

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## 6. Karsilastirma Raporunu Kaydetme

Bu hucre olusan karsilastirma tablosunu diske yazar: `full_comparison.csv` ve `full_comparison.md` dosyalari uretilir.

In [ ]:
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
comparison_csv = RUNS_ROOT / "full_comparison.csv"
comparison_md = RUNS_ROOT / "full_comparison.md"

comparison_df.sort_values(by="effective_batch_size", inplace=True, na_position="last")
comparison_df.to_csv(comparison_csv, index=False)
comparison_md.write_text(comparison_df.to_markdown(index=False), encoding="utf-8")

print(f"[OK] Saved CSV: {comparison_csv}")
print(f"[OK] Saved Markdown: {comparison_md}")
comparison_df

## 7. Sonuclarin Gorsellestirilmesi

Bu hucre basarili kosular icin 4 ana grafigi cizer: best val loss, final val accuracy, toplam sure ve peak VRAM karsilastirmalari.

In [ ]:
plot_df = comparison_df[comparison_df["status"] == "success"].copy()
if len(plot_df) == 0:
    raise ValueError("No successful runs to plot.")

plot_df = plot_df.sort_values("effective_batch_size")
x = plot_df["effective_batch_size"].values

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(x, plot_df["best_val_loss"], marker="o")
axes[0, 0].set_title("effective_batch_size vs best_val_loss")
axes[0, 0].set_xlabel("effective_batch_size")
axes[0, 0].set_ylabel("best_val_loss")

axes[0, 1].plot(x, plot_df["final_val_acc"], marker="o", color="green")
axes[0, 1].set_title("effective_batch_size vs final_val_acc")
axes[0, 1].set_xlabel("effective_batch_size")
axes[0, 1].set_ylabel("final_val_acc")

axes[1, 0].plot(x, plot_df["total_train_time_sec"], marker="o", color="orange")
axes[1, 0].set_title("effective_batch_size vs total_train_time_sec")
axes[1, 0].set_xlabel("effective_batch_size")
axes[1, 0].set_ylabel("total_train_time_sec")

axes[1, 1].plot(x, plot_df["peak_vram_mb"], marker="o", color="red")
axes[1, 1].set_title("effective_batch_size vs peak_vram_mb")
axes[1, 1].set_xlabel("effective_batch_size")
axes[1, 1].set_ylabel("peak_vram_mb")

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Notes

- Bu notebook standart batch training kosusu calistirmaz.
- Sadece accumulation bazli kucuk+büyük EBS karsilastirmasi yapar.
- Konfig degisikligi icin sadece JSON dosyasini guncellemek yeterlidir.